# PROYECTO CENTINELA — FASE 3
## Implementación y Herramientas Profesionales de Deep Learning

**Universidad Santo Tomás · Maestría en Ciencia de Datos**
Profundización I: Redes Neuronales — Deep Learning (EA-N-F-004)

---

### Versión corregida

Este cuaderno incorpora las correcciones derivadas de (a) la retroalimentación de la
Fase 2 y (b) la auditoría interna del primer cuaderno de la Fase 3.

**Principio de trabajo:** ningún número del informe se escribe a mano. Todo sale de
`cifras_informe.json`, que genera el último bloque.

| Bloque | Contenido |
|---|---|
| 0 | Instalación |
| 1 | Carga de datos a disco local + entorno GPU |
| 2 | Pipeline tf.data vs DataLoader (comparación justa) + partición estratificada |
| 3 | Data Augmentation + A/B con y sin |
| 4 | Entrenamiento FP16 + checkpointing |
| 4b | FP32 vs FP16 **medido** |
| 4c | CUDA OOM: provocado y resuelto |
| 5 | Curvas de entrenamiento |
| 6 | Evaluación en test + intervalos de confianza |
| 7 | Optimización Edge: cuantización dinámica vs estática |
| 8 | Exportación .pth / .onnx / .keras + verificación |
| 9 | Ruta de despliegue |
| 10 | Cifras verificadas para el informe |

> **Ejecutar con GPU:** Entorno de ejecución → Cambiar tipo de entorno → T4 GPU.

---
## BLOQUE 0 — Instalación

In [ ]:
!pip install onnx onnxruntime onnxscript -q
!pip install tensorflow -q
print('Librerías instaladas')

---
## BLOQUE 1 — Carga de datos y entorno GPU

**Corrección aplicada.** El cuaderno original leía las imágenes directamente desde
Google Drive, con dos consecuencias: (1) la ruta era la del Drive de un integrante y no
funcionaba desde otra cuenta; (2) Drive tiene latencia alta por archivo, así que el
benchmark del pipeline medía sobre todo la I/O de red, no el diseño del pipeline —
esto explica los 817 ms/batch del cuaderno original.

Ahora los datos se descomprimen en `/content` (SSD local de Colab) y el benchmark
pasa a medir lo que dice medir.

### Cómo cargar los datos — de más rápido a más lento

**Opción A (recomendada) — Google Drive.** Arrastra `datos_centinela_fase3.zip` a
[drive.google.com](https://drive.google.com), a la raíz de *Mi unidad*. Se sube una sola
vez y **sobrevive a los reinicios de sesión**. Esta celda lo monta y lo descomprime.

**Opción B — panel lateral de Colab.** Abre el icono de carpeta (izquierda) y arrastra
el zip a `/content`. Más rápido que el botón de subida, pero se pierde al reiniciar.

**Opción C — botón Choose Files.** Es la más lenta: Colab transporta el archivo
codificado en base64 por el puente de JavaScript. Solo se usa si las anteriores fallan.

La celda busca el archivo en ese orden y usa la primera fuente que encuentre.

In [ ]:
import os, time, zipfile, warnings, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

RUTA_SALIDA = '/content/modelos_fase3'
RUTA_BASE   = '/content/datos_centinela'
NOMBRE_ZIP  = 'datos_centinela_fase3.zip'
os.makedirs(RUTA_SALIDA, exist_ok=True)

def _buscar_zip():
    # 0) ya descomprimido de una corrida anterior
    if os.path.isdir(os.path.join(RUTA_BASE, 'Imagenes_agua', 'train')):
        return 'YA_LISTO'
    # 1) en el disco local de Colab (panel lateral)
    for p in [f'/content/{NOMBRE_ZIP}'] + glob.glob('/content/*.zip'):
        if os.path.exists(p):
            return p
    # 2) en Google Drive: primero el zip, luego la carpeta suelta
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive', force_remount=False)
        for pat in [f'/content/drive/MyDrive/{NOMBRE_ZIP}',
                    f'/content/drive/MyDrive/**/{NOMBRE_ZIP}']:
            hits = glob.glob(pat, recursive=True)
            if hits:
                return hits[0]
        # la carpeta Imagenes_agua subida directamente a Drive
        for pat in ['/content/drive/MyDrive/Imagenes_agua',
                    '/content/drive/MyDrive/**/Imagenes_agua']:
            hits = [h for h in glob.glob(pat, recursive=True) if os.path.isdir(h)]
            if hits:
                return ('CARPETA', hits[0])
    except Exception as err:
        print(f'   (Drive no disponible: {err})')
    return None

origen = _buscar_zip()

if isinstance(origen, tuple) and origen[0] == 'CARPETA':
    import shutil
    src = origen[1]
    print(f'Encontrada la carpeta en Drive: {src}')
    print('Copiando a disco local (Drive es lento leyendo 512 archivos sueltos)...')
    t0 = time.time()
    shutil.copytree(src, os.path.join(RUTA_BASE, 'Imagenes_agua'), dirs_exist_ok=True)
    print(f'Copiado en {time.time()-t0:.1f}s -> {RUTA_BASE}')
elif origen == 'YA_LISTO':
    print(f'Datos ya presentes en {RUTA_BASE} - no hace falta subir nada')
elif origen:
    print(f'Encontrado: {origen}  ({os.path.getsize(origen)/1e6:.1f} MB)')
    t0 = time.time()
    with zipfile.ZipFile(origen) as z:
        z.extractall(RUTA_BASE)
    print(f'Descomprimido en {time.time()-t0:.1f}s -> {RUTA_BASE}')
else:
    print('No se encontro el zip en /content ni en Drive.')
    print('Ultimo recurso: subida por navegador (lenta).')
    print('Alternativa mas rapida: subelo a drive.google.com y vuelve a ejecutar.')
    from google.colab import files
    subidos = files.upload()
    nombre = list(subidos.keys())[0]
    t0 = time.time()
    with zipfile.ZipFile(nombre) as z:
        z.extractall(RUTA_BASE)
    print(f'Descomprimido en {time.time()-t0:.1f}s -> {RUTA_BASE}')

RUTA_IMG   = os.path.join(RUTA_BASE, 'Imagenes_agua', 'train')
RUTA_DATA  = os.path.join(RUTA_BASE, 'data', 'water_dataset.mat')
RUTA_PESOS = os.path.join(RUTA_BASE, 'fase2', 'pesos')

print('\n=== VERIFICACION DE RUTAS ===')
for r, n in [(RUTA_IMG, 'Imagenes'), (RUTA_DATA, 'Dataset .mat'), (RUTA_PESOS, 'Pesos Fase 2')]:
    print(f'   {n:<15} {"OK" if os.path.exists(r) else "FALTA"}')

print('\n=== IMAGENES POR CLASE ===')
total_imgs = 0
for clase in sorted(os.listdir(RUTA_IMG)):
    d = os.path.join(RUTA_IMG, clase)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        total_imgs += n
        print(f'   {clase:<10} {n:>4}')
print(f'   {"TOTAL":<10} {total_imgs:>4}')
assert total_imgs == 512, f'Se esperaban 512 imagenes, hay {total_imgs}'
print('   Conteo correcto (512, igual que en Fase 2)')

import torch, torchvision
import tensorflow as tf
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('\n=== ENTORNO GPU ===')
print(f'GPU disponible:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Modelo GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM total:      {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'VRAM libre:      {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')
print(f'PyTorch:         {torch.__version__}')
print(f'TorchVision:     {torchvision.__version__}')
print(f'TensorFlow:      {tf.__version__}')
print(f'\nDatos en:        {RUTA_BASE}  (disco local, no Drive)')

!nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv

---
## BLOQUE 2 — Pipeline de datos y partición estratificada

**Correcciones aplicadas.**

1. *Benchmark tf.data comparable.* El cuaderno original comparaba un pipeline
   **sin** `.shuffle()` contra uno **con** `.shuffle(buffer=512)`. Ese shuffle obliga a
   leer las 512 imágenes antes del primer batch, así que la medición capturaba el costo
   del barajado, no el del cache — de ahí el −85% reportado. Ahora ambos pipelines
   llevan el mismo shuffle y solo varían `num_parallel_calls`, `.cache()` y `.prefetch()`.

2. *Medición en época 1 y 2 por separado.* El cache solo tiene efecto desde la segunda
   pasada. La época 2 es la comparación válida.

3. *Partición estratificada con índices fijos*, reutilizada en todo el cuaderno, con un
   `assert` que comprueba que train/val/test son disjuntos. Conservar esa salida es la
   lección directa de la Fase 2.

4. **Partición por escenas (la corrección de mayor impacto).** Al auditar la primera
   corrida corregida, *todas* las métricas dieron 100% — incluido un modelo Keras
   entrenado apenas 5 épocas. Un IC95% de [100,0 – 100,0] no es un buen resultado: es un
   síntoma.

   La causa está en los nombres de archivo: `20230802_132524`, `20230802_132821`,
   `20230802_132822`… son **fotos en ráfaga de la misma escena**, tomadas con segundos de
   diferencia. Agrupando por ventanas de 3 s dentro de cada clase, las 512 imágenes son
   en realidad **~145 escenas independientes** (Bersih 95, Keruh 29, Kotor 21), con un
   promedio de 3,5 fotos por escena y un grupo de hasta 128 fotos.

   Un split aleatorio manda la foto de las 13:25:24 a *train* y la de las 13:25:25 a
   *test*: el modelo no generaliza, **reconoce escenas que ya vio**.

   Se construyen por eso **dos particiones** y se reportan **ambos resultados**. La
   diferencia entre las dos *es* la medida de la fuga.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models
from sklearn.model_selection import train_test_split

IMG_SIZE, BATCH_SIZE, SEMILLA = 224, 32, 42
torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASES    = ['Bersih', 'Keruh', 'Kotor']
CLASES_ES = ['Limpia', 'Turbia', 'Contaminada']

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

transform_aug = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),                              # 1
    transforms.RandomHorizontalFlip(p=0.5),                       # 2
    transforms.RandomRotation(degrees=15),                        # 3
    transforms.ColorJitter(brightness=0.2, contrast=0.2,          # 4
                           saturation=0.1, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),     # 5
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

# ---- Particion estratificada 70/15/15 sobre indices fijos ----
dataset_ref = datasets.ImageFolder(root=RUTA_IMG, transform=transform_val)
N_CLASES  = len(dataset_ref.classes)
etiquetas = np.array([y for _, y in dataset_ref.samples])
idx_todos = np.arange(len(dataset_ref))

idx_train, idx_temp = train_test_split(
    idx_todos, test_size=0.30, random_state=SEMILLA, stratify=etiquetas)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=SEMILLA, stratify=etiquetas[idx_temp])

print('=== PARTICION ESTRATIFICADA (indices fijos) ===')
for nombre, idx in [('Train', idx_train), ('Val', idx_val), ('Test', idx_test)]:
    c = np.bincount(etiquetas[idx], minlength=N_CLASES)
    print(f'{nombre:<6} n={len(idx):>4} | ' +
          ' | '.join(f'{CLASES_ES[k]}: {c[k]}' for k in range(N_CLASES)))

assert len(set(idx_train) & set(idx_test)) == 0, 'FUGA: train y test se solapan'
assert len(set(idx_val)   & set(idx_test)) == 0, 'FUGA: val y test se solapan'
print('Comprobacion de fuga: OK - train/val/test disjuntos')

dataset_aug_full = datasets.ImageFolder(root=RUTA_IMG, transform=transform_aug)
ds_tr = Subset(dataset_aug_full, idx_train)
ds_vl = Subset(dataset_ref,      idx_val)
ds_te = Subset(dataset_ref,      idx_test)

loader_train = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, prefetch_factor=2)
loader_val   = DataLoader(ds_vl, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
loader_test  = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# ---- Benchmark PyTorch ----
def medir_tiempo_pt(loader, n_batches=10):
    inicio = time.time()
    for i, (imgs, _) in enumerate(loader):
        imgs = imgs.to(DISPOSITIVO, non_blocking=True)
        if i >= n_batches: break
    return (time.time() - inicio) / n_batches

loader_sin = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=False)
loader_con = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, prefetch_factor=2)

print('\nMidiendo PyTorch DataLoader...')
t_sin = medir_tiempo_pt(loader_sin)
t_con = medir_tiempo_pt(loader_con)
mejora = (t_sin - t_con) / t_sin * 100
print(f'  sin optimizar : {t_sin*1000:.1f} ms/batch')
print(f'  optimizado    : {t_con*1000:.1f} ms/batch  (mejora {mejora:.1f}%)')

# ---- Benchmark tf.data (comparacion justa) ----
AUTOTUNE = tf.data.AUTOTUNE
rutas_imgs = [dataset_ref.samples[i][0] for i in idx_train]
etiq_tf    = [dataset_ref.samples[i][1] for i in idx_train]

def cargar_imagen_tf(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img = (img - tf.constant(IMAGENET_MEAN)) / tf.constant(IMAGENET_STD)
    return img, etiqueta

base_ds = tf.data.Dataset.from_tensor_slices((rutas_imgs, etiq_tf))

# Augmentation equivalente a la de PyTorch, para que la comparacion entre
# frameworks mida EL MISMO TRABAJO (la rubrica pide "el mismo pipeline").
aug_tf = tf.keras.Sequential([
    tf.keras.layers.RandomCrop(IMG_SIZE, IMG_SIZE),   # 1 RandomCrop
    tf.keras.layers.RandomFlip('horizontal'),         # 2 HorizontalFlip
    tf.keras.layers.RandomRotation(15/360.0),         # 3 Rotation 15 grados
    tf.keras.layers.RandomBrightness(0.2),            # 4 ColorJitter (brillo)
    tf.keras.layers.RandomContrast(0.2),              # 4 ColorJitter (contraste)
], name='augmentation')

def cargar_grande_tf(ruta, etiqueta):
    # resize a 256 para que el RandomCrop a 224 tenga margen (igual que PyTorch)
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE + 32, IMG_SIZE + 32])
    img = tf.cast(img, tf.float32) / 255.0
    return img, etiqueta

def aplicar_aug(img, etiqueta):
    img = aug_tf(img, training=True)
    img = (img - tf.constant(IMAGENET_MEAN)) / tf.constant(IMAGENET_STD)
    return img, etiqueta

ds_tf_sin = (base_ds
             .shuffle(len(rutas_imgs), seed=SEMILLA)
             .map(cargar_grande_tf)
             .map(aplicar_aug)
             .batch(BATCH_SIZE))

ds_tf_con = (base_ds
             .shuffle(len(rutas_imgs), seed=SEMILLA)
             .map(cargar_grande_tf, num_parallel_calls=AUTOTUNE)
             .cache()
             .map(aplicar_aug, num_parallel_calls=AUTOTUNE)
             .batch(BATCH_SIZE)
             .prefetch(AUTOTUNE))

print('NOTA: el cache va ANTES del augmentation. Cachear despues congelaria')
print('      las transformaciones aleatorias y anularia el aumento por epoca.')

def medir_epoca_tf(ds):
    inicio, n = time.time(), 0
    for _ in ds: n += 1
    return (time.time() - inicio) / max(n, 1)

print('\nMidiendo tf.data (epoca 1 y epoca 2)...')
t_tf_sin_e1 = medir_epoca_tf(ds_tf_sin)
t_tf_sin_e2 = medir_epoca_tf(ds_tf_sin)
t_tf_con_e1 = medir_epoca_tf(ds_tf_con)
t_tf_con_e2 = medir_epoca_tf(ds_tf_con)

mejora_tf_e1 = (t_tf_sin_e1 - t_tf_con_e1) / t_tf_sin_e1 * 100
mejora_tf_e2 = (t_tf_sin_e2 - t_tf_con_e2) / t_tf_sin_e2 * 100

print(f'\n=== TF.DATA epoca 1 (cache llenandose) ===')
print(f'  sin optimizar: {t_tf_sin_e1*1000:.1f} ms/batch')
print(f'  optimizado   : {t_tf_con_e1*1000:.1f} ms/batch  ({mejora_tf_e1:+.1f}%)')
print(f'=== TF.DATA epoca 2 (cache caliente) <- COMPARACION VALIDA ===')
print(f'  sin optimizar: {t_tf_sin_e2*1000:.1f} ms/batch')
print(f'  optimizado   : {t_tf_con_e2*1000:.1f} ms/batch  ({mejora_tf_e2:+.1f}%)')

print(f'\n{"Framework":<26}{"Sin optim.":>13}{"Optimizado":>13}{"Mejora":>10}')
print('-' * 62)
print(f'{"PyTorch DataLoader":<26}{t_sin*1000:>11.1f}ms{t_con*1000:>11.1f}ms{mejora:>9.1f}%')
print(f'{"tf.data (epoca 1)":<26}{t_tf_sin_e1*1000:>11.1f}ms{t_tf_con_e1*1000:>11.1f}ms{mejora_tf_e1:>9.1f}%')
print(f'{"tf.data (epoca 2)":<26}{t_tf_sin_e2*1000:>11.1f}ms{t_tf_con_e2*1000:>11.1f}ms{mejora_tf_e2:>9.1f}%')

fig, ax = plt.subplots(figsize=(10, 4.5))
etiqs = ['PyTorch\nsin optim', 'PyTorch\noptimizado',
         'tf.data e1\nsin optim', 'tf.data e1\noptimizado',
         'tf.data e2\nsin optim', 'tf.data e2\noptimizado']
vals = [t_sin*1000, t_con*1000, t_tf_sin_e1*1000, t_tf_con_e1*1000,
        t_tf_sin_e2*1000, t_tf_con_e2*1000]
bars = ax.bar(etiqs, vals, color=['#C0392B', '#02C39A']*3, alpha=0.85, width=0.6)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
            f'{v:.0f}ms', ha='center', fontweight='bold', fontsize=9)
ax.set_ylabel('Tiempo por batch (ms)')
ax.set_title('Comparativa de pipelines - mismo shuffle en ambos, epocas 1 y 2',
             fontsize=11, fontweight='bold')
ax.legend(handles=[mpatches.Patch(color='#C0392B', alpha=.85, label='Sin optimizacion'),
                   mpatches.Patch(color='#02C39A', alpha=.85, label='Con optimizacion')])
ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
plt.xticks(fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, 'comparativa_pipelines.png'), dpi=150)
plt.show()

In [ ]:
import re, datetime
from collections import defaultdict

# ---- Agrupamiento por escena: rafagas dentro de una ventana temporal ----
VENTANA_SEG = 3   # fotos de la misma clase a <=3s se consideran la misma escena

def _sello(ruta):
    n = os.path.basename(ruta)
    m = re.match(r'(.+?)_jpg\.rf\.[0-9a-f]+\.jpg$', n)
    b = m.group(1) if m else n
    t = re.match(r'(\d{8})_(\d{6})', b)
    return datetime.datetime.strptime(t.group(1)+t.group(2), '%Y%m%d%H%M%S') if t else None

def construir_grupos(samples, ventana=VENTANA_SEG):
    por_clase = defaultdict(list)
    for i, (ruta, y) in enumerate(samples):
        por_clase[y].append((i, _sello(ruta)))
    grupo = np.empty(len(samples), dtype=int)
    gid = 0
    for y in sorted(por_clase):
        con = sorted([p for p in por_clase[y] if p[1]], key=lambda p: p[1])
        sin = [p for p in por_clase[y] if not p[1]]
        if con:
            gid += 1; grupo[con[0][0]] = gid
            for (ia, ta), (ib, tb) in zip(con, con[1:]):
                if (tb - ta).total_seconds() > ventana:
                    gid += 1
                grupo[ib] = gid
        for i, _ in sin:                 # sin timestamp: cada una es su propia escena
            gid += 1; grupo[i] = gid
    return grupo

grupos = construir_grupos(dataset_ref.samples)
n_escenas = len(np.unique(grupos))
print('=== AGRUPAMIENTO POR ESCENA (rafagas) ===')
print(f'Imagenes            : {len(grupos)}')
print(f'Escenas independientes: {n_escenas}   (ventana {VENTANA_SEG}s)')
for c in range(N_CLASES):
    m = etiquetas == c
    print(f'  {CLASES_ES[c]:<12} {m.sum():>4} fotos en {len(np.unique(grupos[m])):>3} escenas')
tam = np.bincount(grupos)
print(f'Fotos por escena    : promedio {len(grupos)/n_escenas:.1f}, maximo {tam.max()}')
print('\nLECTURA: el dataset NO tiene 512 observaciones independientes.')
print('Un split aleatorio reparte fotos de la MISMA rafaga entre train y test.')

# ---- Particion B: por ESCENA (honesta) ----
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
resto_g, test_g = next(sgkf.split(idx_todos, etiquetas, groups=grupos))
sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
tr_rel, vl_rel = next(sgkf2.split(resto_g, etiquetas[resto_g], groups=grupos[resto_g]))
idx_train_g = resto_g[tr_rel]
idx_val_g   = resto_g[vl_rel]
idx_test_g  = test_g

print('\n=== PARTICION POR ESCENA ===')
for nombre, idx in [('Train', idx_train_g), ('Val', idx_val_g), ('Test', idx_test_g)]:
    c = np.bincount(etiquetas[idx], minlength=N_CLASES)
    print(f'{nombre:<6} n={len(idx):>4} | {len(np.unique(grupos[idx])):>3} escenas | ' +
          ' | '.join(f'{CLASES_ES[k]}: {c[k]}' for k in range(N_CLASES)))

g_tr, g_vl, g_te = (set(grupos[i]) for i in (idx_train_g, idx_val_g, idx_test_g))
assert not (g_tr & g_te), 'FUGA DE ESCENA: una rafaga aparece en train y test'
assert not (g_vl & g_te), 'FUGA DE ESCENA: una rafaga aparece en val y test'
print('Comprobacion de fuga por escena: OK - ninguna rafaga cruza los conjuntos')

# ---- Loaders de la particion honesta (los que usa el resto del cuaderno) ----
ds_tr_g = Subset(dataset_aug_full, idx_train_g)
ds_vl_g = Subset(dataset_ref,      idx_val_g)
ds_te_g = Subset(dataset_ref,      idx_test_g)

loader_train_g = DataLoader(ds_tr_g, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True, prefetch_factor=2)
loader_val_g   = DataLoader(ds_vl_g, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)
loader_test_g  = DataLoader(ds_te_g, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

---
## BLOQUE 3 — Data Augmentation

Cinco transformaciones, elegidas por su semántica en el dominio del agua:

| # | Transformación | Justificación |
|---|---|---|
| 1 | `RandomCrop(224)` tras resize a 256 | Encuadres distintos del mismo cuerpo de agua |
| 2 | `RandomHorizontalFlip` | El agua no tiene orientación izquierda/derecha |
| 3 | `RandomRotation(15°)` | La tablet no siempre se sostiene nivelada |
| 4 | `ColorJitter` leve | Luz de mañana vs mediodía |
| 5 | `GaussianBlur` | Enfoque imperfecto en campo |

**Rechazadas:** `RandomVerticalFlip` (una foto de agua invertida no tiene sentido
físico) y `ColorJitter` fuerte (`saturation=0.3`, `hue=0.1`), porque el color **es** la
señal diagnóstica: alterar el tono puede volver el agua turbia en aparentemente limpia.

**Corrección aplicada:** el aumento se aplica en vivo sobre las imágenes en cada época,
no sobre features precomputadas — señalamiento directo de la Fase 2.

In [ ]:
from PIL import Image

transform_solo_aug = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))])

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
fig.suptitle('Data Augmentation - Original vs Aumentada (x5)', fontsize=13, fontweight='bold')
for row, clase in enumerate(sorted(os.listdir(RUTA_IMG))):
    ruta_clase = os.path.join(RUTA_IMG, clase)
    archivos = sorted([f for f in os.listdir(ruta_clase)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    img = Image.open(os.path.join(ruta_clase, archivos[0])).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    axes[row, 0].imshow(img)
    nombre_es = CLASES_ES[CLASES.index(clase)] if clase in CLASES else clase
    axes[row, 0].set_title(f'{nombre_es}\nOriginal', fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')
    for col in range(1, 6):
        axes[row, col].imshow(transform_solo_aug(img))
        axes[row, col].set_title(f'Aug {col}', fontsize=9)
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, 'augmentation_visual.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}')
print(f'Clases: {dataset_ref.classes} -> {CLASES_ES}')

### 3b — Comparación con y sin augmentation

La guía exige *"comparar el accuracy de validación con y sin aumento tras diez épocas"*.
No se había hecho en el cuaderno original.

Se reporta el resultado que salga. Con 512 imágenes y transfer learning el efecto puede
ser pequeño o incluso negativo; ese también es un resultado válido.

In [ ]:
def crear_resnet18(n_clases):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, n_clases))
    return m

def entrenar_corto(loader_tr, epocas=10, usar_amp=True, etiqueta='', loader_ev=None):
    # Entrena desde cero. Devuelve (mejor_acc_val, vram_pico_MB, seg/epoca).
    if loader_ev is None: loader_ev = loader_val_g
    torch.manual_seed(SEMILLA)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m = crear_resnet18(N_CLASES).to(DISPOSITIVO)
    opt = torch.optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda') if usar_amp else None
    mejor, tiempos = 0.0, []
    for _ in range(epocas):
        m.train(); t0 = time.time()
        for imgs, y in loader_tr:
            imgs, y = imgs.to(DISPOSITIVO), y.to(DISPOSITIVO)
            opt.zero_grad()
            if usar_amp:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    loss = crit(m(imgs), y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = crit(m(imgs), y); loss.backward(); opt.step()
        torch.cuda.synchronize(); tiempos.append(time.time() - t0)
        m.eval(); c = t = 0
        with torch.no_grad():
            for imgs, y in loader_ev:
                imgs, y = imgs.to(DISPOSITIVO), y.to(DISPOSITIVO)
                c += (m(imgs).argmax(1) == y).sum().item(); t += y.size(0)
        mejor = max(mejor, c / t)
    vram = torch.cuda.max_memory_allocated() / 1e6
    print(f'  {etiqueta:<28} acc_val={mejor*100:5.1f}% | VRAM={vram:6.0f}MB | {np.mean(tiempos):.2f}s/epoca')
    del m, opt; torch.cuda.empty_cache()
    return mejor, vram, float(np.mean(tiempos))

loader_sin_aug = DataLoader(Subset(dataset_ref, idx_train_g), batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=2, pin_memory=True)

print('=== A/B DE DATA AUGMENTATION (10 epocas, particion por escena) ===')
acc_sin_aug, _, _ = entrenar_corto(loader_sin_aug,  10, True, 'SIN augmentation', loader_val_g)
acc_con_aug, _, _ = entrenar_corto(loader_train_g,  10, True, 'CON augmentation', loader_val_g)
print(f'\nDiferencia: {(acc_con_aug-acc_sin_aug)*100:+.1f} pp a favor de '
      f'{"CON" if acc_con_aug > acc_sin_aug else "SIN"} augmentation')

### 3c — Medición de la fuga por escenas

Se entrena **el mismo modelo, con los mismos hiperparámetros**, sobre las dos
particiones, y se evalúa cada uno en su propio conjunto de prueba.

Si el split aleatorio da mucho más que el split por escena, la diferencia no es
capacidad del modelo: es **memorización de escenas vistas en entrenamiento**.

Esta es la auditoría central de la Fase 3.

In [ ]:
def entrenar_y_probar(idx_tr, idx_vl, idx_te, etiqueta, epocas=15):
    ltr = DataLoader(Subset(dataset_aug_full, idx_tr), batch_size=BATCH_SIZE,
                     shuffle=True, num_workers=2, pin_memory=True)
    lvl = DataLoader(Subset(dataset_ref, idx_vl), batch_size=BATCH_SIZE,
                     shuffle=False, num_workers=2, pin_memory=True)
    lte = DataLoader(Subset(dataset_ref, idx_te), batch_size=BATCH_SIZE,
                     shuffle=False, num_workers=2, pin_memory=True)
    torch.manual_seed(SEMILLA); torch.cuda.empty_cache()
    m = crear_resnet18(N_CLASES).to(DISPOSITIVO)
    opt = torch.optim.Adam(m.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(); sc = torch.amp.GradScaler('cuda')
    for _ in range(epocas):
        m.train()
        for imgs, y in ltr:
            imgs, y = imgs.to(DISPOSITIVO), y.to(DISPOSITIVO)
            opt.zero_grad()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = crit(m(imgs), y)
            sc.scale(loss).backward(); sc.step(opt); sc.update()
    m.eval(); c = t = 0
    with torch.no_grad():
        for imgs, y in lte:
            imgs, y = imgs.to(DISPOSITIVO), y.to(DISPOSITIVO)
            c += (m(imgs).argmax(1) == y).sum().item(); t += y.size(0)
    a = c / t
    print(f'  {etiqueta:<34} acc_test = {a*100:5.1f}%   (n={t})')
    del m, opt; torch.cuda.empty_cache()
    return a

print('=== DEMOSTRACION DE LA FUGA POR ESCENAS ===')
print('Mismo modelo, mismos hiperparametros, 15 epocas. Solo cambia el split.\n')
acc_split_aleatorio = entrenar_y_probar(idx_train,   idx_val,   idx_test,
                                        'Split ALEATORIO (con fuga)')
acc_split_escena    = entrenar_y_probar(idx_train_g, idx_val_g, idx_test_g,
                                        'Split POR ESCENA (honesto)')

brecha = (acc_split_aleatorio - acc_split_escena) * 100
print(f'\n{"="*62}')
print(f'BRECHA DE FUGA: {brecha:+.1f} puntos porcentuales')
print(f'{"="*62}')
print('El split aleatorio reparte fotos de la misma rafaga entre train y test.')
print('Esa diferencia es la porcion del desempeno que NO es generalizacion,')
print(f'sino reconocimiento de escenas ya vistas ({n_escenas} escenas reales,')
print(f'no {len(grupos)} observaciones independientes).')
print('\nTodas las metricas que siguen usan la particion POR ESCENA.')

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Split aleatorio\n(con fuga)', 'Split por escena\n(honesto)'],
              [acc_split_aleatorio*100, acc_split_escena*100],
              color=['#C0392B', '#02C39A'], alpha=0.85, width=0.5)
for b, v in zip(bars, [acc_split_aleatorio*100, acc_split_escena*100]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{v:.1f}%',
            ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Accuracy en test (%)'); ax.set_ylim(0, 109)
ax.set_title(f'Efecto de la fuga por escenas (brecha {brecha:+.1f} pp)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=.3); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, 'fuga_por_escenas.png'), dpi=150)
plt.show()

---
## BLOQUE 4 — Entrenamiento en GPU con precisión mixta

ResNet-18 con transfer learning desde ImageNet. Precisión mixta FP16 con `GradScaler`,
`ReduceLROnPlateau`, early stopping y checkpointing cada 5 épocas.

**Por qué ResNet-18 y no ResNet-50:** 11,2 M parámetros frente a 25,6 M. El destino es
una tablet sin GPU; el modelo más liviano que resuelve la tarea es el correcto.

**Los checkpoints guardan `state_dict` + estado del optimizador + `mejor_loss_val`.**
Sin el optimizador no se puede retomar el entrenamiento: Adam pierde sus momentos y el
reinicio introduce un salto en la trayectoria.

In [ ]:
EPOCAS, PACIENCIA, LR = 20, 8, 1e-4

modelo = crear_resnet18(N_CLASES).to(DISPOSITIVO)
print(f'ResNet-18: {sum(p.numel() for p in modelo.parameters()):,} parametros')

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

criterio    = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=LR, weight_decay=1e-4)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode='min',
                                                         patience=4, factor=0.5)
scaler = torch.amp.GradScaler('cuda')

mejor_loss, sin_mejora = float('inf'), 0
historial = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
vram_fp16_list, tiempo_lista = [], []
ruta_mejor = os.path.join(RUTA_SALIDA, 'mejor_resnet18.pth')

print(f'\nEntrenando con precision mixta FP16 | Epocas {EPOCAS} | LR {LR} | Batch {BATCH_SIZE}')
print('-' * 78)

for epoca in range(1, EPOCAS + 1):
    t_inicio = time.time()
    modelo.train(); lt = ct = tt = 0
    for imgs, etiq in loader_train_g:
        imgs, etiq = imgs.to(DISPOSITIVO), etiq.to(DISPOSITIVO)
        optimizador.zero_grad()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = modelo(imgs); loss = criterio(logits, etiq)
        scaler.scale(loss).backward(); scaler.step(optimizador); scaler.update()
        lt += loss.item(); ct += (logits.argmax(1) == etiq).sum().item(); tt += etiq.size(0)
    loss_t, acc_t = lt / len(loader_train_g), ct / tt
    torch.cuda.synchronize(); t_epoca = time.time() - t_inicio

    modelo.eval(); lv = cv = tv = 0
    with torch.no_grad():
        for imgs, etiq in loader_val_g:
            imgs, etiq = imgs.to(DISPOSITIVO), etiq.to(DISPOSITIVO)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                logits = modelo(imgs); loss = criterio(logits, etiq)
            lv += loss.item(); cv += (logits.argmax(1) == etiq).sum().item(); tv += etiq.size(0)
    loss_v, acc_v = lv / len(loader_val_g), cv / tv

    for k, v in [('train_loss', loss_t), ('val_loss', loss_v),
                 ('train_acc', acc_t), ('val_acc', acc_v)]:
        historial[k].append(v)
    vram = torch.cuda.max_memory_allocated() / 1e6
    vram_fp16_list.append(vram); tiempo_lista.append(t_epoca)
    scheduler.step(loss_v)

    if loss_v < mejor_loss:
        mejor_loss, sin_mejora = loss_v, 0
        torch.save(modelo.state_dict(), ruta_mejor)
    else:
        sin_mejora += 1

    if epoca % 5 == 0:
        rc = os.path.join(RUTA_SALIDA, f'checkpoint_epoca_{epoca:02d}.pth')
        torch.save({'epoca': epoca, 'state_dict': modelo.state_dict(),
                    'optimizador': optimizador.state_dict(),
                    'mejor_loss_val': mejor_loss}, rc)
        print(f'   Checkpoint guardado: {os.path.basename(rc)}')

    print(f'Epoca {epoca:2d}/{EPOCAS} | Loss {loss_t:.4f}/{loss_v:.4f} | '
          f'Acc {acc_t*100:.1f}%/{acc_v*100:.1f}% | VRAM {vram:.0f}MB | '
          f'{t_epoca:.1f}s | sin mejora {sin_mejora}/{PACIENCIA}')

    if sin_mejora >= PACIENCIA:
        print(f'Early stopping en epoca {epoca}'); break

modelo.load_state_dict(torch.load(ruta_mejor, map_location=DISPOSITIVO))
print(f'\nMejor modelo cargado (loss val: {mejor_loss:.4f})')

### 4b — FP32 vs FP16: **medido**, no estimado

**Corrección aplicada.** El cuaderno original no midió FP32: calculaba
`VRAM_FP32 = VRAM_FP16 × 1.8` y `t_FP32 = t_FP16 × 1.4`, y el informe reportaba esos
productos como si fueran datos. La rúbrica pide el ahorro **documentado**.

Una época cuesta ~6 s, así que medirlo de verdad es cuestión de un minuto de cómputo.

In [ ]:
print('=== FP32 vs FP16 - MEDICION DIRECTA (5 epocas cada uno) ===')
_, vram_fp32_med, t_fp32_med = entrenar_corto(loader_train_g, 5, False, 'FP32 (sin AMP)')
_, vram_fp16_med, t_fp16_med = entrenar_corto(loader_train_g, 5, True,  'FP16 (con AMP)')

ahorro_vram = (1 - vram_fp16_med / vram_fp32_med) * 100
acel = t_fp32_med / t_fp16_med

print(f'\n{"Metrica":<26}{"FP32":>12}{"FP16":>12}{"Cambio":>14}')
print('-' * 64)
print(f'{"VRAM pico (MB)":<26}{vram_fp32_med:>12.0f}{vram_fp16_med:>12.0f}{ahorro_vram:>13.1f}%')
print(f'{"Tiempo/epoca (s)":<26}{t_fp32_med:>12.2f}{t_fp16_med:>12.2f}{acel:>13.2f}x')
print('\nAmbas cifras son MEDIDAS. El ahorro de VRAM es menor al 50% porque AMP')
print('mantiene los pesos maestros en FP32; el ahorro esta en las activaciones.')

fig2, ax2 = plt.subplots(1, 2, figsize=(10, 4))
fig2.suptitle('FP32 vs FP16 - valores MEDIDOS en Tesla T4', fontsize=12, fontweight='bold')
for a, (tit, vals, uni) in zip(ax2, [
        ('VRAM pico (MB)', [vram_fp32_med, vram_fp16_med], 'MB'),
        ('Tiempo por epoca (s)', [t_fp32_med, t_fp16_med], 's')]):
    bars = a.bar(['FP32', 'FP16'], vals, color=['#C0392B', '#02C39A'], alpha=0.85)
    for b, v in zip(bars, vals):
        a.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
               f'{v:.1f}{uni}', ha='center', fontweight='bold')
    a.set_title(tit, fontweight='bold'); a.grid(axis='y', alpha=.3)
    a.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, 'comparativa_fp32_fp16.png'), dpi=150)
plt.show()

### 4c — CUDA OOM: provocado y resuelto

El nivel Estratégico del criterio 2 pide *"resuelve CUDA OOM con evidencia"*. Se provoca
deliberadamente subiendo el tamaño de lote hasta agotar la VRAM, y se documenta la
mitigación.

In [ ]:
print('=== GESTION DE VRAM: PROVOCAR Y RESOLVER UN CUDA OOM ===')
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
modelo_oom = crear_resnet18(N_CLASES).to(DISPOSITIVO)
opt_oom  = torch.optim.Adam(modelo_oom.parameters(), lr=1e-4)
crit_oom = nn.CrossEntropyLoss()

batch_max, oom_en = None, None
for bs in [256, 512, 1024, 2048, 4096]:
    try:
        x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=DISPOSITIVO)
        y = torch.randint(0, N_CLASES, (bs,), device=DISPOSITIVO)
        opt_oom.zero_grad(); crit_oom(modelo_oom(x), y).backward(); opt_oom.step()
        print(f'  batch={bs:<5} OK    | VRAM pico: {torch.cuda.max_memory_allocated()/1e6:7.0f} MB')
        batch_max = bs
        del x, y; torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError as e:
        oom_en = bs
        print(f'  batch={bs:<5} CUDA OOM  <- {str(e).splitlines()[0][:70]}')
        torch.cuda.empty_cache(); break

if oom_en:
    print(f'\nOOM reproducido en batch={oom_en}. Mitigaciones:')
    print(f'  1. Reducir el batch (el proyecto usa {BATCH_SIZE})')
    print( '  2. Precision mixta FP16 -> menos memoria en activaciones')
    print( '  3. Acumulacion de gradientes -> batch efectivo grande, memoria de uno chico')
    print( '  4. torch.utils.checkpoint -> recomputa activaciones en el backward')
    print(f'\nVerificacion de 1+2+3 (batch efectivo {oom_en} con AMP):')
    try:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        sc = torch.amp.GradScaler('cuda'); acum = 4; sub = oom_en // acum
        opt_oom.zero_grad()
        for _ in range(acum):
            x = torch.randn(sub, 3, IMG_SIZE, IMG_SIZE, device=DISPOSITIVO)
            y = torch.randint(0, N_CLASES, (sub,), device=DISPOSITIVO)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = crit_oom(modelo_oom(x), y) / acum
            sc.scale(loss).backward(); del x, y
        sc.step(opt_oom); sc.update()
        print(f'  RESUELTO: batch efectivo {oom_en} via {acum} pasos de {sub} + AMP')
        print(f'  VRAM pico: {torch.cuda.max_memory_allocated()/1e6:.0f} MB')
    except torch.cuda.OutOfMemoryError:
        print('  Aun OOM: reducir mas el sub-batch.')
else:
    print(f'\nSin OOM hasta batch={batch_max}. Se documenta el limite empirico alcanzado.')

del modelo_oom, opt_oom; torch.cuda.empty_cache()

---
## BLOQUE 5 — Curvas de entrenamiento

In [ ]:
ep = range(1, len(historial['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Curvas de entrenamiento - ResNet-18 con FP16 (Tesla T4)',
             fontsize=13, fontweight='bold')
axes[0].plot(ep, historial['train_loss'], '--', color='#065A82', alpha=.7, label='Train')
axes[0].plot(ep, historial['val_loss'], '-', color='#065A82', lw=2, label='Val')
axes[0].set_title('Perdida'); axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss')
axes[1].plot(ep, [a*100 for a in historial['train_acc']], '--', color='#02C39A', alpha=.7, label='Train')
axes[1].plot(ep, [a*100 for a in historial['val_acc']], '-', color='#02C39A', lw=2, label='Val')
axes[1].set_title('Precision'); axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Accuracy (%)')
for a in axes:
    a.legend(); a.grid(alpha=.3); a.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, 'curvas_entrenamiento_f3.png'), dpi=150)
plt.show()

---
## BLOQUE 6 — Evaluación en test con intervalos de confianza

**Corrección aplicada.** El docente señaló en la Fase 2: *"Reporten la incertidumbre.
Para el ROC-AUC de la rama visual, den un intervalo de confianza en vez de cuatro
decimales sobre 78 imágenes"*. Se añade bootstrap con 2000 remuestreos.

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, classification_report)

def evaluar(m, loader, dispositivo, usar_amp=True):
    m.eval(); yt, yp, ypr = [], [], []
    with torch.no_grad():
        for imgs, y in loader:
            imgs = imgs.to(dispositivo)
            if usar_amp and dispositivo.type == 'cuda':
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    logits = m(imgs)
            else:
                logits = m(imgs)
            logits = logits.float()
            ypr.extend(F.softmax(logits, 1).cpu().numpy())
            yp.extend(logits.argmax(1).cpu().numpy())
            yt.extend(y.numpy())
    return np.array(yt), np.array(yp), np.array(ypr)

def ic_bootstrap(y_true, y_pred, y_prob, n=2000, alfa=0.05):
    rng, accs, aucs = np.random.default_rng(SEMILLA), [], []
    N = len(y_true)
    for _ in range(n):
        s = rng.integers(0, N, N)
        if len(np.unique(y_true[s])) < len(np.unique(y_true)): continue
        accs.append((y_true[s] == y_pred[s]).mean())
        try:
            aucs.append(roc_auc_score(y_true[s], y_prob[s], multi_class='ovr', average='macro'))
        except ValueError:
            pass
    q = [100*alfa/2, 100*(1-alfa/2)]
    return np.percentile(accs, q), np.percentile(aucs, q)

y_true, y_pred, y_prob = evaluar(modelo, loader_test_g, DISPOSITIVO)
acc = (y_true == y_pred).mean()
auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
(ic_acc_lo, ic_acc_hi), (ic_auc_lo, ic_auc_hi) = ic_bootstrap(y_true, y_pred, y_prob)

print(f'=== RESULTADOS EN TEST (n={len(y_true)}) ===')
print(f'Accuracy: {acc*100:.1f}%   IC95% [{ic_acc_lo*100:.1f}% - {ic_acc_hi*100:.1f}%]')
print(f'ROC-AUC : {auc:.3f}      IC95% [{ic_auc_lo:.3f} - {ic_auc_hi:.3f}]')
print(f'\nEl intervalo es ancho porque n={len(y_true)}. Reportar cuatro decimales')
print('sobre esta muestra seria falsa precision.\n')
print(classification_report(y_true, y_pred, target_names=CLASES_ES, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASES_ES).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de confusion - ResNet-18 (test n={len(y_true)})\n'
             f'Acc {acc*100:.1f}% IC95%[{ic_acc_lo*100:.0f}-{ic_acc_hi*100:.0f}%]',
             fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, 'matriz_confusion_f3.png'), dpi=150)
plt.show()

---
## BLOQUE 7 — Optimización Edge: cuantización

**Dos correcciones críticas respecto del cuaderno original.**

**1. El accuracy INT8 de "100%" no medía nada.** El código evaluaba sobre
`ImageFolder(RUTA_IMG)` sin barajar y cortaba en 4 batches. Como `Bersih` (Limpia) tiene
311 imágenes, esas 128 eran **todas de una sola clase** y **todas del conjunto de
entrenamiento**. El informe lo reportaba como "+2,6 pp" de mejora. La cuantización no
puede mejorar el accuracy: un resultado que contradice la teoría es señal de error de
medición. Ahora se evalúa sobre el **mismo test** que el FP32.

**2. La cuantización era un no-op.** `quantize_dynamic` con `qconfig_spec={nn.Linear}`
sobre ResNet-18 alcanza **una sola** capa (512→3 = 1.539 parámetros, el **0,014%** del
modelo). Todo lo demás son `Conv2d`, que esta técnica no toca — de ahí el 0% de
reducción. El informe lo atribuía a "los metadatos compensan el ahorro", explicación
incorrecta. Se añade **cuantización estática PTQ**, que sí cuantiza las convoluciones, y
se conserva la dinámica como contraste documentado.

La latencia se mide como **mediana de 30 corridas tras 10 de calentamiento**.

In [ ]:
import copy
import torch.ao.quantization as tq
from torchvision.models.quantization import resnet18 as qresnet18

torch.backends.quantized.engine = 'fbgemm'
CPU = torch.device('cpu')

def tam_mb(m, ruta):
    torch.save(m.state_dict(), ruta)
    return os.path.getsize(ruta) / 1e6

def medir_latencia(m, n=30, descarte=10):
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        for _ in range(descarte): m(x)
        ts = []
        for _ in range(n):
            t0 = time.perf_counter(); m(x); ts.append((time.perf_counter()-t0)*1000)
    return float(np.median(ts))

def acc_en_test(m, disp=CPU):
    yt, yp, _ = evaluar(m, loader_test_g, disp, usar_amp=False)
    return (yt == yp).mean()

# --- A. FP32 base ---
modelo_fp32 = copy.deepcopy(modelo).to(CPU).eval()
tam_fp32 = tam_mb(modelo_fp32, os.path.join(RUTA_SALIDA, 'modelo_fp32.pth'))
lat_fp32 = medir_latencia(modelo_fp32)
acc_fp32 = acc_en_test(modelo_fp32)

# --- B. Cuantizacion DINAMICA (contraste documentado) ---
modelo_din = tq.quantize_dynamic(copy.deepcopy(modelo_fp32),
                                 qconfig_spec={nn.Linear}, dtype=torch.qint8)
tam_din = tam_mb(modelo_din, os.path.join(RUTA_SALIDA, 'modelo_int8_dinamico.pth'))
lat_din = medir_latencia(modelo_din)
acc_din = acc_en_test(modelo_din)

n_linear = sum(p.numel() for mo in modelo_fp32.modules()
               if isinstance(mo, nn.Linear) for p in mo.parameters())
n_total = sum(p.numel() for p in modelo_fp32.parameters())
print('=== POR QUE LA CUANTIZACION DINAMICA NO REDUJO EL TAMANO ===')
print(f'Parametros en capas Linear : {n_linear:>12,}')
print(f'Parametros totales         : {n_total:>12,}')
print(f'Fraccion cuantizable       : {n_linear/n_total*100:>12.3f}%')
print('quantize_dynamic solo alcanza Linear/LSTM/GRU. ResNet-18 es casi toda')
print('Conv2d, asi que la tecnica no aplica. Ese es el motivo real del 0%.\n')

# --- C. Cuantizacion ESTATICA (PTQ con calibracion) ---
modelo_est = qresnet18(weights=None, quantize=False)
modelo_est.fc = nn.Linear(512, N_CLASES)
sd = modelo.state_dict()
sd_dst = {k: v for k, v in sd.items() if not k.startswith('fc.')}
sd_dst['fc.weight'] = sd['fc.1.weight']; sd_dst['fc.bias'] = sd['fc.1.bias']
res = modelo_est.load_state_dict(sd_dst, strict=False)
print(f'Carga de pesos -> faltantes: {len(res.missing_keys)}, inesperados: {len(res.unexpected_keys)}')

modelo_est.eval()
modelo_est.fuse_model(is_qat=False)
modelo_est.qconfig = tq.get_default_qconfig('fbgemm')
tq.prepare(modelo_est, inplace=True)

print('Calibrando con imagenes de TRAIN (nunca de test)...')
loader_calib = DataLoader(Subset(dataset_ref, idx_train_g[:128]), batch_size=32, shuffle=False)
with torch.no_grad():
    for imgs, _ in loader_calib: modelo_est(imgs)
tq.convert(modelo_est, inplace=True)

tam_est = tam_mb(modelo_est, os.path.join(RUTA_SALIDA, 'modelo_int8_estatico.pth'))
lat_est = medir_latencia(modelo_est)
acc_est = acc_en_test(modelo_est)

print('\n' + '=' * 74)
print(f'TABLA DE OPTIMIZACION EDGE - mismo test para las tres columnas (n={len(idx_test_g)})')
print('=' * 74)
print(f'{"Metrica":<26}{"FP32":>13}{"INT8 dinam.":>15}{"INT8 estat.":>15}')
print('-' * 74)
print(f'{"Tamano (MB)":<26}{tam_fp32:>13.1f}{tam_din:>15.1f}{tam_est:>15.1f}')
print(f'{"Latencia mediana (ms)":<26}{lat_fp32:>13.1f}{lat_din:>15.1f}{lat_est:>15.1f}')
print(f'{"Accuracy en test (%)":<26}{acc_fp32*100:>13.1f}{acc_din*100:>15.1f}{acc_est*100:>15.1f}')
print('=' * 74)
print(f'Reduccion de tamano   FP32->INT8 estatico: {(1-tam_est/tam_fp32)*100:>6.1f}%')
print(f'Reduccion de latencia FP32->INT8 estatico: {(1-lat_est/lat_fp32)*100:>6.1f}%')
print(f'Cambio de accuracy    FP32->INT8 estatico: {(acc_est-acc_fp32)*100:>+6.1f} pp')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Optimizacion Edge - FP32 vs INT8 dinamico vs INT8 estatico',
             fontsize=12, fontweight='bold')
for a, (tit, vals) in zip(axes, [
        ('Tamano (MB)',   [tam_fp32, tam_din, tam_est]),
        ('Latencia (ms)', [lat_fp32, lat_din, lat_est]),
        ('Accuracy (%)',  [acc_fp32*100, acc_din*100, acc_est*100])]):
    bars = a.bar(['FP32', 'INT8\ndinam.', 'INT8\nestat.'], vals,
                 color=['#C0392B', '#E8A33D', '#02C39A'], alpha=0.85)
    for b, v in zip(bars, vals):
        a.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
               f'{v:.1f}', ha='center', fontweight='bold')
    a.set_title(tit, fontweight='bold'); a.grid(axis='y', alpha=.3)
    a.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_SALIDA, 'optimizacion_edge.png'), dpi=150)
plt.show()

---
## BLOQUE 8 — Exportación y verificación

**Correcciones aplicadas.**

1. *Ruta estable explícita.* La bitácora original afirmaba haber rechazado `dynamo=True`
   y usado "la ruta estable", pero el log mostraba
   `[torch.onnx] Obtain model graph ... with torch.export.export(...)`, que **es** dynamo.
   En PyTorch 2.11 el exportador dynamo es el **predeterminado**: hay que pasar
   `dynamo=False` explícitamente.

2. *Archivo único.* Por defecto los pesos van a un sidecar `.onnx.data` y el `.onnx`
   queda en ~90 KB — **no carga si se entrega solo**. Se fuerza
   `save_as_external_data=False`.

3. *El `.keras` se guardaba sin entrenar.* El código creaba ResNet50V2 con cabeza
   aleatoria y lo salvaba sin llamar a `fit()`. Como la ruta de despliegue declara que el
   `.tflite` sale de ese `.keras`, se estaría desplegando un modelo sin entrenar.

4. *Verificación sobre logits*, no sobre softmax (que comprime diferencias y hace la
   prueba más permisiva), y también sobre imágenes reales de test.

In [ ]:
import onnx, onnxruntime as ort

modelo_export = copy.deepcopy(modelo).to(CPU).eval()
X_dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

# --- 1. .pth ---
ruta_pth = os.path.join(RUTA_SALIDA, 'Centinela_Fase3_ResNet18.pth')
torch.save({'state_dict': modelo_export.state_dict(), 'arquitectura': 'ResNet18',
            'n_clases': N_CLASES, 'clases': CLASES_ES, 'img_size': IMG_SIZE,
            'acc_test': float(acc), 'auc_test': float(auc),
            'ic95_acc': [float(ic_acc_lo), float(ic_acc_hi)]}, ruta_pth)
print(f'1. .pth   {os.path.getsize(ruta_pth)/1e6:>6.1f} MB')

# --- 2. .onnx por la RUTA ESTABLE explicita ---
ruta_onnx = os.path.join(RUTA_SALIDA, 'Centinela_Fase3_ResNet18.onnx')
try:
    torch.onnx.export(modelo_export, X_dummy, ruta_onnx, export_params=True,
                      opset_version=18, input_names=['entrada'], output_names=['salida'],
                      dynamic_axes={'entrada': {0: 'lote'}, 'salida': {0: 'lote'}},
                      dynamo=False)
    print('   Exportado por la ruta estable (dynamo=False)')
except TypeError:
    torch.onnx.export(modelo_export, X_dummy, ruta_onnx, export_params=True,
                      opset_version=18, input_names=['entrada'], output_names=['salida'],
                      dynamic_axes={'entrada': {0: 'lote'}, 'salida': {0: 'lote'}})
    print('   Esta version no acepta el argumento dynamo; se uso el default')

_m = onnx.load(ruta_onnx)
onnx.save_model(_m, ruta_onnx, save_as_external_data=False)
onnx.checker.check_model(onnx.load(ruta_onnx))
if os.path.exists(ruta_onnx + '.data'):
    os.remove(ruta_onnx + '.data')
print(f'2. .onnx  {os.path.getsize(ruta_onnx)/1e6:>6.1f} MB (archivo unico, valido)')

# --- 3. .keras ENTRENADO ---
def crear_keras(n_clases, img_size=224):
    base = tf.keras.applications.ResNet50V2(include_top=False, weights='imagenet',
                                            input_shape=(img_size, img_size, 3))
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dropout(0.3)(x)
    return tf.keras.Model(base.input, tf.keras.layers.Dense(n_clases, activation='softmax')(x))

def ds_keras(indices, barajar):
    rutas = [dataset_ref.samples[i][0] for i in indices]
    ys    = [dataset_ref.samples[i][1] for i in indices]
    d = tf.data.Dataset.from_tensor_slices((rutas, ys))
    if barajar: d = d.shuffle(len(rutas), seed=SEMILLA)
    return (d.map(cargar_imagen_tf, num_parallel_calls=AUTOTUNE)
             .cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))

modelo_keras = crear_keras(N_CLASES)
modelo_keras.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('\n3. Entrenando la implementacion Keras (5 epocas)...')
modelo_keras.fit(ds_keras(idx_train_g, True), validation_data=ds_keras(idx_val_g, False),
                 epochs=5, verbose=2)
loss_k, acc_k = modelo_keras.evaluate(ds_keras(idx_test_g, False), verbose=0)
ruta_keras = os.path.join(RUTA_SALIDA, 'Centinela_Fase3_ResNet.keras')
modelo_keras.save(ruta_keras)
print(f'   .keras {os.path.getsize(ruta_keras)/1e6:>6.1f} MB | acc test Keras: {acc_k*100:.1f}%')
print('   NOTA: .pth y .keras son DOS IMPLEMENTACIONES EQUIVALENTES')
print('   (ResNet-18 PyTorch / ResNet50V2 Keras), no el mismo objeto.')

# --- Verificacion de consistencia ---
with torch.no_grad():
    logits_torch = modelo_export(X_dummy).numpy()
sess = ort.InferenceSession(ruta_onnx)
logits_onnx = sess.run(None, {'entrada': X_dummy.numpy()})[0]
consistente = np.allclose(logits_torch, logits_onnx, atol=1e-4)
diff_max = float(np.abs(logits_torch - logits_onnx).max())

print(f'\n=== VERIFICACION DE CONSISTENCIA (sobre LOGITS) ===')
print(f'PyTorch vs ONNX   : {"CONSISTENTE" if consistente else "INCONSISTENTE"}')
print(f'Diferencia maxima : {diff_max:.2e}   (umbral atol=1e-4)')

imgs_reales, _ = next(iter(loader_test_g))
with torch.no_grad():
    lt = modelo_export(imgs_reales).numpy()
lo = sess.run(None, {'entrada': imgs_reales.numpy()})[0]
print(f'Sobre {len(imgs_reales)} imagenes reales: '
      f'{"CONSISTENTE" if np.allclose(lt, lo, atol=1e-4) else "INCONSISTENTE"} '
      f'(dif. max {np.abs(lt-lo).max():.2e})')
print(f'Acuerdo de predicciones: {(lt.argmax(1) == lo.argmax(1)).mean()*100:.1f}%')

---
## BLOQUE 9 — Ruta de despliegue

**Rama visual (CNN) — offline en la tablet.** ResNet-18 cuantizado INT8 → LiteRT desde
el modelo Keras. Foto del río → Limpia / Turbia / Contaminada. Funciona sin internet.

**Rama temporal (GRU) — API REST en la nube.** GRU de la Fase 2 servido con ONNX Runtime.
14 días de pH → probabilidad de riesgo.

**Por qué separarlas:** los modelos recurrentes tienen operadores que no siempre están
soportados en LiteRT y cuyo comportamiento varía entre versiones del runtime.

> **Advertencia honesta.** La *fusión* de ambas ramas, tal como se implementó en la
> Fase 2, no está validada: la clave de alineación quedó constante y la modalidad visual
> no llegó a participar. Hasta rehacer esa ablación, el sistema se documenta como **dos
> señales presentadas al operador por separado**, no como un modelo fusionado.

In [ ]:
print('=' * 66)
print('RUTA DE DESPLIEGUE - PROYECTO CENTINELA FASE 3')
print('=' * 66)
print(f'''
ESCENARIO: Junta Administradora de Acueducto Rural (JAAR)
           Conectividad intermitente en campo

RAMA VISUAL (CNN) - OFFLINE en tablet
  Modelo      : ResNet-18 cuantizado INT8 (estatico)
  Formato     : LiteRT (.tflite) convertido desde .keras
  Tamano      : {tam_est:.1f} MB   Latencia: {lat_est:.1f} ms/imagen (CPU)
  Accuracy    : {acc_est*100:.1f}% en test por escena (n={len(idx_test_g)})
  Flujo       : Foto del agua -> Limpia / Turbia / Contaminada
  Ventaja     : Funciona sin internet

RAMA TEMPORAL (GRU) - API REST en la nube
  Modelo      : GRU entrenado en Fase 2
  Formato     : ONNX Runtime en servidor
  Flujo       : 14 dias de pH -> probabilidad de riesgo
  Razon       : LSTM/GRU no son confiables en LiteRT

FLUJO POR CONECTIVIDAD
  Sin internet : solo CNN en tablet -> alerta visual inmediata
  Con internet : CNN + GRU -> dos senales complementarias al operador
  NOTA         : la fusion de Fase 2 no esta validada; no se despliega
                 como modelo unico hasta rehacer la ablacion

DESPLIEGUE RESPONSABLE
  Transparencia    : la JAAR sabe que el sistema usa IA entrenada con 512
                     imagenes y que sus predicciones son orientativas
  Beneficio social : deteccion proactiva sin laboratorios ni conectividad
                     permanente; de dias a segundos
  Falso positivo   : falsa alarma -> monitoreo extra. Costo bajo
  Falso negativo   : no avisar riesgo real -> salud comprometida. Costo ALTO
  Decision         : umbrales conservadores, se prioriza el recall.
                     El sistema complementa al operador, no lo reemplaza
''')
print('=' * 66)

---
## BLOQUE 10 — Cifras verificadas para el informe

**Regla de trabajo:** si un número no aparece en esta salida, no se reporta en el
informe. Es la lección directa de la Fase 2 — *"el informe tiene que contar exactamente
lo que hace el código"*.

In [ ]:
import json

resumen = {
    'GPU': torch.cuda.get_device_name(0),
    'PyTorch': torch.__version__,
    'TensorFlow': tf.__version__,
    'SPLIT_n_train': len(idx_train_g), 'SPLIT_n_val': len(idx_val_g),
    'SPLIT_n_test': len(idx_test_g),
    'SPLIT_escenas_totales': int(n_escenas),
    'SPLIT_ventana_agrupamiento_s': VENTANA_SEG,
    'SPLIT_escenas_train': int(len(np.unique(grupos[idx_train_g]))),
    'SPLIT_escenas_test': int(len(np.unique(grupos[idx_test_g]))),
    'FUGA_acc_split_aleatorio_pct': round(acc_split_aleatorio*100, 1),
    'FUGA_acc_split_escena_pct': round(acc_split_escena*100, 1),
    'FUGA_brecha_pp': round(brecha, 1),
    'PIPELINE_pytorch_sin_ms': round(t_sin*1000, 1),
    'PIPELINE_pytorch_con_ms': round(t_con*1000, 1),
    'PIPELINE_pytorch_mejora_pct': round(mejora, 1),
    'PIPELINE_tfdata_e2_sin_ms': round(t_tf_sin_e2*1000, 1),
    'PIPELINE_tfdata_e2_con_ms': round(t_tf_con_e2*1000, 1),
    'PIPELINE_tfdata_e2_mejora_pct': round(mejora_tf_e2, 1),
    'AUG_acc_val_sin_pct': round(acc_sin_aug*100, 1),
    'AUG_acc_val_con_pct': round(acc_con_aug*100, 1),
    'GPU_vram_fp32_MB': round(vram_fp32_med),
    'GPU_vram_fp16_MB': round(vram_fp16_med),
    'GPU_ahorro_vram_pct': round(ahorro_vram, 1),
    'GPU_t_fp32_s': round(t_fp32_med, 2),
    'GPU_t_fp16_s': round(t_fp16_med, 2),
    'GPU_aceleracion_x': round(acel, 2),
    'GPU_epocas_corridas': len(historial['train_loss']),
    'TEST_particion': 'por escena (sin fuga de rafagas)',
    'TEST_accuracy_pct': round(acc*100, 1),
    'TEST_ic95_accuracy': f'[{ic_acc_lo*100:.1f} - {ic_acc_hi*100:.1f}]',
    'TEST_roc_auc': round(auc, 3),
    'TEST_ic95_roc_auc': f'[{ic_auc_lo:.3f} - {ic_auc_hi:.3f}]',
    'EDGE_tam_fp32_MB': round(tam_fp32, 1),
    'EDGE_tam_int8_dinamico_MB': round(tam_din, 1),
    'EDGE_tam_int8_estatico_MB': round(tam_est, 1),
    'EDGE_lat_fp32_ms': round(lat_fp32, 1),
    'EDGE_lat_int8_dinamico_ms': round(lat_din, 1),
    'EDGE_lat_int8_estatico_ms': round(lat_est, 1),
    'EDGE_acc_fp32_pct': round(acc_fp32*100, 1),
    'EDGE_acc_int8_dinamico_pct': round(acc_din*100, 1),
    'EDGE_acc_int8_estatico_pct': round(acc_est*100, 1),
    'EDGE_pct_params_cuantizables_dinamico': round(n_linear/n_total*100, 3),
    'EXPORT_consistencia_onnx': bool(consistente),
    'EXPORT_dif_max_onnx': f'{diff_max:.2e}',
    'EXPORT_acc_keras_pct': round(acc_k*100, 1),
}

print('=' * 62)
print('CIFRAS VERIFICADAS PARA EL INFORME - FASE 3')
print('=' * 62)
for k, v in resumen.items():
    print(f'{k:<40}{v}')
print('=' * 62)

with open(os.path.join(RUTA_SALIDA, 'cifras_informe.json'), 'w', encoding='utf8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print('\nGuardado: cifras_informe.json  <- unica fuente de verdad del informe')

---
## Descargar resultados

In [ ]:
import shutil
shutil.make_archive('/content/resultados_fase3', 'zip', RUTA_SALIDA)
print(f'ZIP: {os.path.getsize("/content/resultados_fase3.zip")/1e6:.1f} MB')
print('\nArchivos generados:')
for f in sorted(os.listdir(RUTA_SALIDA)):
    print(f'   {f:<45} {os.path.getsize(os.path.join(RUTA_SALIDA, f))/1e6:>7.1f} MB')

from google.colab import files
files.download('/content/resultados_fase3.zip')